# Prediksi Biaya Asuransi Kesehatan dengan Multiple Linear Regression

**Dataset:** Medical Cost Personal Dataset — Kaggle  
**Sumber:** https://www.kaggle.com/datasets/mirichoi0218/insurance

## Tujuan
Membangun model regresi untuk memperkirakan `charges` (biaya asuransi kesehatan) berdasarkan usia, jenis kelamin, BMI, jumlah anak, status merokok, dan wilayah.

> Catatan: model ini untuk pembelajaran dan eksplorasi dataset, bukan alat untuk menentukan premi atau keputusan medis nyata.

## 1. Import library
Jalankan sel secara berurutan. Jika Google Colab meminta izin, lanjutkan sesuai instruksi.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

## 2. Memuat dataset
Unduh file `insurance.csv` dari halaman Kaggle di atas, lalu unggah ke Colab saat diminta.

In [ ]:
from google.colab import files
uploaded = files.upload()

# Membaca file CSV yang diunggah
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Unggah file CSV, misalnya insurance.csv.")
df = pd.read_csv(csv_files[0])

print("Ukuran dataset:", df.shape)
display(df.head())

## 3. Memahami dan memeriksa data

In [ ]:
print("Informasi kolom:")
df.info()

print("\nJumlah nilai kosong per kolom:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nStatistik deskriptif:")
display(df.describe(include="all").T)

print("\nJumlah baris duplikat:", df.duplicated().sum())

## 4. Pembersihan data
Baris dengan nilai kosong dihapus. Baris duplikat diperiksa dan dihapus agar tidak terhitung berulang.

In [ ]:
df = df.dropna().drop_duplicates().copy()

expected_columns = {"age", "sex", "bmi", "children", "smoker", "region", "charges"}
missing_columns = expected_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Kolom berikut tidak ditemukan: {missing_columns}")

print("Ukuran setelah pembersihan:", df.shape)

## 5. Eksplorasi dan visualisasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["charges"], kde=True, ax=axes[0])
axes[0].set_title("Distribusi Biaya Asuransi")
axes[0].set_xlabel("Charges")
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=0.7, ax=axes[1])
axes[1].set_title("Usia dan Biaya Asuransi berdasarkan Status Merokok")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x="smoker", y="charges", ax=axes[0])
axes[0].set_title("Biaya Asuransi berdasarkan Status Merokok")
sns.boxplot(data=df, x="region", y="charges", ax=axes[1])
axes[1].set_title("Biaya Asuransi berdasarkan Wilayah")
plt.tight_layout()
plt.show()

print("Rata-rata charges berdasarkan status merokok:")
display(df.groupby("smoker")["charges"].agg(["count", "mean", "median"]).round(2))

## 6. Menyiapkan fitur dan target
Target yang diprediksi adalah `charges`. Kolom kategori (`sex`, `smoker`, dan `region`) diubah menjadi kolom numerik dengan One-Hot Encoding.

In [ ]:
X = df[["age", "sex", "bmi", "children", "smoker", "region"]]
y = df["charges"]

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Data latih:", X_train.shape)
print("Data uji:", X_test.shape)

## 7. Melatih model Multiple Linear Regression

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 8. Evaluasi model
- **MAE**: rata-rata selisih absolut antara prediksi dan nilai sebenarnya; makin kecil makin baik.
- **RMSE**: memberi penalti lebih besar pada kesalahan yang besar; makin kecil makin baik.
- **R²**: proporsi variasi target yang dapat dijelaskan model; makin mendekati 1 umumnya makin baik.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metrik": ["MAE", "RMSE", "R²"],
    "Nilai": [mae, rmse, r2]
})
display(metrics.round(3))

print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

## 9. Membandingkan nilai aktual dan prediksi

In [ ]:
comparison = pd.DataFrame({
    "Aktual": y_test.to_numpy(),
    "Prediksi": y_pred
})
display(comparison.head(10).round(2))

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Biaya aktual")
plt.ylabel("Biaya prediksi")
plt.title("Nilai Aktual vs Prediksi")
plt.tight_layout()
plt.show()

## 10. Contoh prediksi untuk data baru
Ubah nilai pada contoh berikut sesuai skenario yang ingin dicoba. Hasil prediksi bukan penawaran asuransi resmi.

In [ ]:
contoh = pd.DataFrame([{
    "age": 30,
    "sex": "female",
    "bmi": 25.5,
    "children": 1,
    "smoker": "no",
    "region": "southwest"
}])

prediksi_biaya = model.predict(contoh)[0]
print(f"Perkiraan charges untuk contoh data: ${prediksi_biaya:,.2f}")

## 11. Kesimpulan otomatis untuk bahan laporan
Isi angka berdasarkan hasil yang muncul setelah seluruh sel dijalankan.

In [ ]:
print("RINGKASAN HASIL")
print(f"- Jumlah data setelah pembersihan: {len(df)}")
print(f"- Jumlah data uji: {len(X_test)}")
print(f"- MAE: ${mae:,.2f}")
print(f"- RMSE: ${rmse:,.2f}")
print(f"- R²: {r2:.4f}")

print("\nPanduan interpretasi:")
print("- MAE dan RMSE menyatakan besarnya kesalahan prediksi dalam satuan charges (USD).")
print("- R² menunjukkan proporsi variasi charges yang dapat dijelaskan oleh model.")
print("- Periksa grafik dan rata-rata kelompok untuk membahas pola data; jangan menyimpulkan sebab-akibat hanya dari korelasi.")

## Referensi
1. Choi, M. *Medical Cost Personal Datasets*. Kaggle. https://www.kaggle.com/datasets/mirichoi0218/insurance
2. Lantz, B. (2019). *Machine Learning with R: Expert techniques for predictive modeling*. Packt Publishing.
3. Scikit-learn documentation. *LinearRegression* dan *Model evaluation*. https://scikit-learn.org/stable/